# Phase 2A: Complete One-Shot Colab Pipeline for Infant Cry Classification

Run this notebook top-to-bottom in Google Colab. It handles package upload/unzip, Drive setup, dependency install, strict data verification, auxiliary cry-domain adaptation, AST and Whisper download/caching, resumable feature extraction, ablation feature-set creation, classifier training, fixed-split testing, and repeated-split evaluation.

This notebook is configured for a one-shot full run: original AST, auxiliary-adapted AST, Whisper, handcrafted features, and the main ablation combinations are all produced in the same pass.

## 0. Mount Drive, Upload Zip, and Locate Project

You only need `phase2a_colab_package.zip`, created locally with `python scripts/package_phase2a_colab.py`. This notebook will unzip it into Drive if the project folder is not already there. You do not need the raw 5GB dataset zip in Colab.

In [ ]:
import os
import zipfile
from pathlib import Path

try:
    from google.colab import drive, files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive') if IN_COLAB else Path.cwd().parent
PROJECT_DIR = DRIVE_ROOT / 'Infant-State-Recognition-System'
PACKAGE_NAME = 'phase2a_colab_package.zip'

if not (PROJECT_DIR / 'src' / 'phase2a').exists():
    candidate = DRIVE_ROOT / PACKAGE_NAME
    if not candidate.exists() and IN_COLAB:
        print(f'Upload {PACKAGE_NAME}. This is the minimal package, not the raw 5GB dataset zip.')
        uploaded = files.upload()
        uploaded_name = next(iter(uploaded))
        candidate = Path(uploaded_name)
    assert candidate.exists(), f'Missing package zip: {candidate}'
    print('Unzipping package into project folder on Drive...')
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(candidate, 'r') as z:
        z.extractall(PROJECT_DIR)

assert (PROJECT_DIR / 'src' / 'phase2a').exists(), f'Project not found after unzip: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())
print('Project files are on Drive and will persist across Colab reconnects.')

## 1. Install Dependencies

In [ ]:
!pip install -q librosa==0.10.1 soundfile==0.12.1 transformers accelerate scikit-learn pandas scipy tqdm joblib matplotlib seaborn

import os
HF_CACHE = PROJECT_DIR / 'data_lake' / 'cache' / 'huggingface'
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE / 'transformers')
print('Dependencies installed')
print('HuggingFace model cache:', HF_CACHE)

## 2. Verify Strict Phase A Data

In [ ]:
import sys
from pathlib import Path
import pandas as pd

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

manifest_path = PROJECT_DIR / 'data_lake' / 'manifests' / 'cause5_authentic_manifest_v2.csv'
unified_path = PROJECT_DIR / 'data_lake' / 'manifests' / 'unified_manifest_v2.csv'
assert manifest_path.exists(), f'Missing manifest: {manifest_path}'
assert unified_path.exists(), f'Missing unified manifest: {unified_path}'

df = pd.read_csv(manifest_path)
unified = pd.read_csv(unified_path)
aux = unified[(unified['include_for_aux_pretraining'] == True) & (unified['dedupe_status'] == 'kept') & (unified['canonical_label'] != 'unknown_or_uncertain')]
needed_paths = set(df['processed_path']).union(set(aux['processed_path']))
missing = [p for p in needed_paths if not (PROJECT_DIR / p).exists() and not Path(p).exists()]

print('Strict 5-class rows:', len(df))
display(df['canonical_label'].value_counts())
print('Approved auxiliary binary-adaptation rows:', len(aux))
display(aux['canonical_label'].value_counts())
print('Unique processed audio files needed:', len(needed_paths))
print('Missing needed audio files:', len(missing))
assert not missing, f'Missing {len(missing)} required audio files. First missing: {missing[0]}'

## 3. Create Reproducible Split

In [ ]:
from src.phase2a.data import load_or_create_splits

split_df = load_or_create_splits(manifest_path)
display(pd.crosstab(split_df['split'], split_df['canonical_label']))
print('Split manifest saved beside the Phase A manifest.')

## 4. Auxiliary Cry-Domain Adaptation

This uses only safe auxiliary cry/non-cry rows from `unified_manifest_v2.csv`. It does **not** use weak rows as 5-class labels. The one-shot run trains a head-only AST binary adapter and later compares original AST vs auxiliary-adapted AST on the strict 5-class task.

In [ ]:
RUN_AUXILIARY_ADAPTATION = True
AUX_EPOCHS = 3
AUX_BATCH_SIZE = 8
AUX_UNFREEZE_LAST_N = 1  # one-shot run: adapt the last AST block so embeddings actually change.

ADAPTED_AST_DIR = PROJECT_DIR / 'results' / 'phase2a' / 'auxiliary_adaptation' / 'best_ast_binary_adapter'

if RUN_AUXILIARY_ADAPTATION and not ADAPTED_AST_DIR.exists():
    !python scripts/phase2a_auxiliary_adaptation.py --epochs {AUX_EPOCHS} --batch-size {AUX_BATCH_SIZE} --unfreeze-last-n {AUX_UNFREEZE_LAST_N}
elif ADAPTED_AST_DIR.exists():
    print('Found existing adapted AST checkpoint, skipping auxiliary adaptation.')
else:
    print('Auxiliary adaptation disabled; using original AST only.')

USE_ADAPTED_AST = ADAPTED_AST_DIR.exists()
print('Use adapted AST for additional feature extraction:', USE_ADAPTED_AST)
print('Adapted AST directory:', ADAPTED_AST_DIR)

## 5. Extract Feature Bank

Start with AST. This is the most important expensive step, and the output is cached under `data_lake/features/phase2a/`.

In [ ]:
# One-shot full run: original AST + Whisper + handcrafted, then adapted AST if available.
EMBEDDING_MODELS = ['ast', 'whisper']
FORCE_REEXTRACT = False

base_cmd = 'python scripts/phase2a_feature_bank.py --checkpoint-every 25 --models ' + ' '.join(EMBEDDING_MODELS)
if FORCE_REEXTRACT:
    base_cmd += ' --force'
print('Extracting original pretrained embeddings and handcrafted features:')
print(base_cmd)
!{base_cmd}

if USE_ADAPTED_AST:
    adapted_cmd = f'python scripts/phase2a_feature_bank.py --checkpoint-every 25 --models ast --ast-model-name-or-path {ADAPTED_AST_DIR}'
    if FORCE_REEXTRACT:
        adapted_cmd += ' --force'
    print('Extracting auxiliary-adapted AST embeddings:')
    print(adapted_cmd)
    !{adapted_cmd}
else:
    raise RuntimeError('Auxiliary-adapted AST was not created. The one-shot full run expects it.')

## 6. Build Ablation Feature Sets

This creates combined feature banks so we can answer: what happens without auxiliary adaptation, without Whisper, with Whisper, and with both auxiliary adaptation + Whisper.

In [ ]:
!python scripts/phase2a_build_feature_sets.py
print('Ablation feature sets created.')

## 7. Train Classifiers and Ensemble

In [ ]:
!python scripts/phase2a_train_classifiers.py
print('Classifier training complete')

## 8. Repeated-Split Evaluation

This reruns training/testing across multiple stratified splits on each cached feature bank. It is slower than the fixed split, but it gives a more reliable score range.

In [ ]:
RUN_REPEATED_EVAL = True
REPEATED_EVAL_REPEATS = 5

if RUN_REPEATED_EVAL:
    feature_dir = PROJECT_DIR / 'data_lake' / 'features' / 'phase2a'
    feature_files = sorted(feature_dir.glob('*_embeddings.npz')) + sorted(feature_dir.glob('*_features.npz'))
    print('Feature banks for repeated evaluation:')
    for feature_file in feature_files:
        print(' -', feature_file.name)
    for feature_file in feature_files:
        !python scripts/phase2a_repeated_evaluation.py --feature-file {feature_file} --repeats {REPEATED_EVAL_REPEATS}
else:
    print('Repeated evaluation disabled.')

## 9. Inspect Results

In [ ]:
import json
from pathlib import Path

metrics_dir = PROJECT_DIR / 'results' / 'phase2a' / 'metrics'
print('Metrics directory:', metrics_dir)

for path in sorted(metrics_dir.glob('*.json')):
    print('\n' + '=' * 80)
    print(path.name)
    metrics = json.loads(path.read_text())

    if 'repeated_eval' in path.name:
        print('Repeated evaluation summary:')
        for model_name, row in metrics.get('summary', {}).items():
            print(
                f"{model_name}: mean={row['test_macro_f1_mean']:.4f}, "
                f"std={row['test_macro_f1_std']:.4f}, "
                f"min={row['test_macro_f1_min']:.4f}, "
                f"max={row['test_macro_f1_max']:.4f}"
            )
        continue

    items = metrics.items() if 'all_feature_sets' in path.name else [(path.stem, metrics)]
    for feature_set, models in items:
        for model_name, result in models.items():
            if not isinstance(result, dict) or 'test' not in result:
                continue
            test = result['test']
            print(f"{feature_set} | {model_name}: macro_f1={test['macro_f1']:.4f}, balanced_acc={test['balanced_accuracy']:.4f}, mcc={test['mcc']:.4f}")

## 10. Output Files

This one-shot run writes everything under `results/phase2a/` and `data_lake/features/phase2a/`: auxiliary adapter checkpoint, original AST features, adapted AST features, Whisper features, handcrafted features, combined ablation feature sets, fixed-split metrics, repeated-split metrics, predictions, and trained classifier artifacts.

## 11. Phase 3 — Edge Distillation onto EfficientAT (Both Variants)

This block distils the Phase 2A teacher (RBF SVM on the AST-aux + Whisper + handcrafted feature bank) into a real **EfficientAT MobileNetV3** student from `github.com/fschmid56/EfficientAT`. The backbone is AudioSet-pretrained (already includes the `baby_cry` class) with ImageNet pretraining underneath, so the student starts with a strong audio prior at a fraction of AST's size.

We run **both** variants with the **same 5-seed repeated-split protocol** Phase 2A used, so the numbers are directly comparable:

- `mn10_as`: **4.88 M params** — RPi 4 / mobile sweet spot
- `mn04_as`: **0.98 M params** — microcontroller-class (ESP32-S3)

For each seed we (a) re-stratify train/val/test from the strict 5-class manifest, (b) build a validation-weighted **multi-teacher Phase 2A ensemble** from the cached feature banks (`AST`, `AST-aux`, `Whisper`, `handcrafted`, and combined branches), (c) distil the student with KD (KL at temperature T) + label-smoothed CE + class-balanced sampling + mixup + SpecAugment, and (d) evaluate on that seed's held-out test split. EfficientAT was trained at 32 kHz so audio is resampled 16 kHz → 32 kHz on the fly. Per-seed checkpoints are saved and automatically resumed if Colab disconnects.

Outputs per variant:

- `results/phase2a/metrics/phase3_edge_student_<variant>_repeated_eval.json` — mean ± std macro-F1, balanced acc, per-class F1, teacher/student agreement, retention vs teacher.
- `results/phase2a/metrics/phase3_edge_student_<variant>_metrics.json` — best-seed detail with FP32 / INT8 size and CPU / GPU latency.
- `results/phase2a/predictions/phase3_edge_student_<variant>_test_predictions.csv` — best-seed test predictions for downstream confusion-matrix / report figures.
- `results/phase2a/artifacts/phase3_edge_student_<variant>_{fp32,int8}.pt` — deployable checkpoints.

In [ ]:
RUN_EDGE_DISTILLATION = True
EDGE_N_SEEDS = 5
EDGE_EPOCHS = 35
EDGE_BATCH_SIZE = 24
EDGE_TEACHER_TOP_N = 8
EDGE_TEACHER_RANDOM_CANDIDATES = 2000
EDGE_EARLY_STOP_PATIENCE = 8

if RUN_EDGE_DISTILLATION:
    !pip install -q torchaudio

    print('\n' + '=' * 80)
    print('Phase 3 :: distilling EfficientAT mn10_as (4.88M params)')
    print('=' * 80)
    !python scripts/phase3_distill_edge_student.py \
        --variant mn10_as \
        --n-seeds {EDGE_N_SEEDS} \
        --epochs {EDGE_EPOCHS} \
        --batch-size {EDGE_BATCH_SIZE} \
        --teacher-top-n {EDGE_TEACHER_TOP_N} \
        --teacher-random-candidates {EDGE_TEACHER_RANDOM_CANDIDATES} \
        --early-stop-patience {EDGE_EARLY_STOP_PATIENCE}

    print('\n' + '=' * 80)
    print('Phase 3 :: distilling EfficientAT mn04_as (0.98M params)')
    print('=' * 80)
    !python scripts/phase3_distill_edge_student.py \
        --variant mn04_as \
        --n-seeds {EDGE_N_SEEDS} \
        --epochs {EDGE_EPOCHS} \
        --batch-size {EDGE_BATCH_SIZE} \
        --teacher-top-n {EDGE_TEACHER_TOP_N} \
        --teacher-random-candidates {EDGE_TEACHER_RANDOM_CANDIDATES} \
        --early-stop-patience {EDGE_EARLY_STOP_PATIENCE}
else:
    print('Skipping Phase 3 edge distillation.')

## 12. Phase 3 — Generate Report Figures

Renders all publication-quality figures (headline progression, feature-bank comparison, confusion matrix, per-class breakdown, 3- vs 5-class, auxiliary curve, edge-target overview, confidence distribution) into `reports/phase3_report/figures/` and `reports/phase3_presentation/`. Reads from `results/phase2a/`. Skips silently if a metric file is missing, so it is safe to run partway through the pipeline.

In [ ]:
!python scripts/phase3_make_figures.py --results-dir results/phase2a